# 实时波形网页（双模式保留）
同时保留两种输入模式并攻克**实时**：
1. 独立网页(后台 HTTP 服务) + 浏览器 canvas 实时滚动波形 + BPM + 收/停按钮。
2. `MODE` 开关切 MIC / LINE_IN，各走各的 Codec 驱动(R5/R7 不同)。

> ⚠ **I2C 超时处理**：若 Cell1 报 `TimeoutError`，说明 I2C 总线被上一次中断的会话卡死，**只有整板断电重新上电**能释放(仅 Restart Kernel / 软重启无效)。断电重启后从 Cell1 重跑。

顺序：Cell1 配置 codec -> Cell2 起 DMA -> Cell3 定义网页HTML -> Cell4 起后台采集+网页服务并打印 URL，在本机/PYNQ浏览器打开 URL 即可实时看波形并点开始/停止录音。

In [ ]:
\
import time, smbus2

# ================= 输入模式开关 =================
# 'MIC'     : 耳机/MIC 收音(TRRS -> codec LINP/LINN, 当前用这条)
# 'LINE_IN' : 电脑 3.5mm 双头线 -> 板卡 LINE_IN 音频输入 (0x400B/0x400D=0x07, 官方 libpynq 配置)
MODE = 'MIC'              # 等电脑转接线到货后, 改成 'LINE_IN' 重新跑本 cell 即可
INPUT_SEL = {
    'MIC':     {'MIX':0x11, 'R57':0x10},   # 混合器使能 + MIC(20dB LDBOOST)
    'LINE_IN': {'MIX':0x01, 'R57':0x07},   # 官方 select_line_in(): R4/R6=0x01, R5/R7=0x07
}[MODE]

import time, smbus2, os

GAIN = 0x10                # 输入增益(0x10 约 -30dB, 已验证的中等音量工作窗口)
CAD  = 0x3B                # ADAU1761

# ===== 自动定位 codec 所在 I2C 总线(探测 0x3B, 不再硬编码) =====
def _find_bus():
    avail = sorted(int(f[4:]) for f in os.listdir('/dev') if f.startswith('i2c-'))
    print('发现 I2C 总线:', avail)
    if not avail:
        print('!! /dev 下没有 i2c-* 设备(驱动/内核未加载 i2c)')
        return None
    for no in avail:
        b = None
        try:
            b = smbus2.SMBus(no)
            b.write_byte(CAD, 0x00)   # 与 WORK.ipynb 一致的探测: 仅需 ACK
            print('总线%d: 0x3B 有 ACK 响应 -> codec 在这' % no)
            return no
        except OSError as e:
            print('总线%d: 0x3B errno=%d (%s)' % (no, e.errno, e.strerror))
        except Exception as e:
            print('总线%d: %s: %s' % (no, type(e).__name__, e))
        finally:
            try:
                if b is not None: b.close()
            except Exception: pass
    return None

# 残留句柄/自愈型卡死: 一次探测全失败则重试最多3轮
for _try in range(3):
    BUS = _find_bus()
    if BUS is not None: break
    print('第%d轮探测未发现 0x3B, 稍后重试...' % (_try+1))
    time.sleep(0.5)
else:
    BUS = None

BUS = _find_bus()
assert BUS is not None, '未找到 0x3B(codec): 请确认供电/接线或整板断电重启'
_bus_ = {'b': smbus2.SMBus(BUS)}

def wr16(r, v, retry=4):   # SMBus 块写通路; 超时自动重开总线重试(不能走 i2c_rdwr)
    for _ in range(retry):
        try:
            _bus_['b'].write_i2c_block_data(CAD, r>>8, [r&0xFF, v]); time.sleep(0.002); return
        except Exception:
            try: _bus_['b'].close()
            except Exception: pass
            time.sleep(0.3)
            try: _bus_['b'] = smbus2.SMBus(BUS)
            except Exception: pass
    raise TimeoutError('codec I2C 连续超时(总线%d) -> 需整板断电重新上电(仅重启Kernel无效)' % BUS)

wr16(0x4000,0x00); wr16(0x4000,0x03)
wr16(0x40F9,0x7F); wr16(0x40FA,0x01)
wr16(0x4015,0x00); wr16(0x4016,0x00); wr16(0x4017,0x00)
for r,v in [(0x400A,INPUT_SEL['MIX']),(0x400B,INPUT_SEL['R57']),
            (0x400C,INPUT_SEL['MIX']),(0x400D,INPUT_SEL['R57']),
            (0x400E,GAIN),(0x400F,GAIN),(0x4010,0x01),(0x4026,0x33)]:
    wr16(r, v)
# 配置后再探测一次, 确认总线仍活(codec 未被 I2C 写崩)
try:
    _bus_['b'].write_byte(CAD, 0x00); alive = 'OK'
except Exception:
    alive = 'DEAD'
    try:
        _bus_['b'].close()
    except Exception:
        pass
print('[codec] MODE=%s 总线=%d MIX=0x%02X R5/R7=0x%02X GAIN=0x%02X 探测=%s'
      % (MODE, BUS, INPUT_SEL['MIX'], INPUT_SEL['R57'], GAIN, alive))


In [ ]:
\
import time, struct
import numpy as np
from pynq import Overlay, allocate

LEN = 512                 # 已验证: 该长度 DMA 能 idle+刷新
Fs  = 46300.0
ol = Overlay('design_1.bit')
seg, bpm, dma = ol.segment_stream_0, ol.bpm_detect_0, ol.axi_dma_0
buf = allocate(shape=(LEN,), dtype=np.uint32)
dma.recvchannel.start()

def grab_block():
    """采一个 512 块(已验证 idle+刷新), 返回左声道 int32 数组(256点)"""
    try: dma.recvchannel.transfer(buf)
    except Exception: pass
    buf.flush()
    seg.mmio.write(0x00, LEN)
    time.sleep(0.03)
    buf.invalidate()
    ra = np.asarray(buf).copy()
    return (ra[::2] ^ 0x800000).astype(np.int32) - 0x800000

def read_reg(off):
    try: return bpm.mmio.read(off)
    except Exception: return 0

def read_bpm():
    try: return struct.unpack('<f', int(read_reg(0x40)).to_bytes(4,'little'))[0]
    except Exception: return 0.0

print('[dma] LEN=%d Fs=%.0f ready  (BPM@0x40 cnt@0x50 env@0x60)' % (LEN, Fs))


In [ ]:
HTML_PAGE = r'''<!doctype html><html lang="zh"><head><meta charset="utf-8">
<title>PYNQ 实时收音 - 波形 + BPM</title>
<style>
 html,body{margin:0;height:100%;background:#0d1117;color:#e6edf3;font-family:Segoe UI,Arial,monospace}
 #wrap{max-width:1200px;margin:0 auto;padding:16px}
 h1{font-size:20px;margin:6px 0 14px;display:inline-block}
 #status{font-size:14px;color:#8b949e;float:right;line-height:34px}
 .cards{display:flex;gap:12px;margin-bottom:14px;flex-wrap:wrap}
 .card{background:#161b22;border:1px solid #30363d;border-radius:8px;padding:10px 16px;min-width:120px;flex:1}
 .card .k{font-size:12px;color:#8b949e}
 .card .v{font-size:30px;font-weight:700;margin-top:4px}
 #bpm{color:#f0883e}
 #swing{color:#3fb950}
 #peaks{color:#58a6ff}
 canvas{background:#0d1117;border:1px solid #30363d;border-radius:8px;display:block}
 #cv{width:100%;height:260px}
 #bpmcv{width:100%;height:90px;margin-top:12px}
 .row{display:flex;gap:12px;margin-top:12px;flex-wrap:wrap}
 button{background:#238636;color:#fff;border:none;border-radius:6px;padding:10px 22px;font-size:16px;cursor:pointer}
 button.off{background:#da3633}
 .note{margin-top:14px;font-size:12px;color:#8b949e;line-height:1.7}
</style></head><body><div id="wrap">
<h1>PYNQ-Z2 实时收音</h1><span id="status">连接中…</span>
<div class="cards">
  <div class="card"><div class="k">BPM</div><div class="v" id="bpm">--</div></div>
  <div class="card"><div class="k">Swing(幅值)</div><div class="v" id="swing">0</div></div>
  <div class="card"><div class="k">Mode</div><div class="v" id="peaks">--</div></div>
</div>
<div><button id="btn">Start Recording</button>
<span style="margin-left:12px;font-size:13px;color:#8b949e" id="modeinfo"></span></div>
<canvas id="cv"></canvas>
<canvas id="bpmcv"></canvas>
<div class="note">波形 = 最近 ~1.5s 采集样本(实时滚动)。</div>
</div>
<script>
const cv=document.getElementById('cv'), cx=cv.getContext('2d');
const bcv=document.getElementById('bpmcv'), bx=bcv.getContext('2d');
let running=true, bpm=0;
let sw=0, bpmHist=new Array(240).fill(0);
function resize(){cv.width=cv.clientWidth*window.devicePixelRatio;cv.height=cv.clientHeight*window.devicePixelRatio;
bcv.width=bcv.clientWidth*window.devicePixelRatio;bcv.height=bcv.clientHeight*window.devicePixelRatio;}
window.addEventListener('resize',resize);resize();
const btn=document.getElementById('btn');
btn.onclick=()=>{ running=!running;
  fetch('/api/'+(running?'start':'stop'),{method:'POST'});
  btn.textContent=running?'Stop Recording':'Start Recording';
  btn.className=running?'':'off'; };
function drawWave(w){
  resize();
  const W=cv.width,H=cv.height,g=cx;
  g.clearRect(0,0,W,H); g.strokeStyle='#30363d'; g.lineWidth=1;
  g.beginPath(); g.moveTo(0,H/2); g.lineTo(W,H/2); g.stroke();
  if(!w||w.length<2) return;
  let mn=w[0],mx=w[0]; for(const v of w){if(v<mn)mn=v;if(v>mx)mx=v;}
  const span=(mx-mn)||1, mid=(mx+mn)/2;
  const step=Math.max(1,Math.floor(w.length/W));
  const n=Math.floor(w.length/step);
  g.beginPath(); g.strokeStyle='#58a6ff'; g.lineWidth=1.5;
  for(let i=0;i<n;i++){
    const x=i/(n-1)*W;
    const y=H/2-(w[i*step]-mid)/span*(H*0.86);
    i===0?g.moveTo(x,y):g.lineTo(x,y);
  }
  g.stroke();
}
function drawBPM(){
  const W=bcv.width,H=bcv.height,g=bx;
  g.clearRect(0,0,W,H); g.strokeStyle='#30363d'; g.lineWidth=1;
  g.beginPath(); g.moveTo(0,H/2); g.lineTo(W,H/2); g.stroke();
  g.strokeStyle='#f0883e'; g.lineWidth=1.5; g.beginPath();
  for(let i=0;i<bpmHist.length;i++){
    const x=i/(bpmHist.length-1)*W;
    const y=H-(bpmHist[i]/240)*H;
    i===0?g.moveTo(x,y):g.lineTo(x,y);
  }
  g.stroke();
}
async function tick(){
  try{
    const r=await fetch('/api/state',{cache:'no-store'});
    const d=await r.json();
    document.getElementById('status').textContent = d.running?'REC ●':'STOPPED ○';
    document.getElementById('bpm').textContent = d.bpm>0?d.bpm.toFixed(1):'0';
    document.getElementById('swing').textContent = d.swing;
    document.getElementById('peaks').textContent = d.mode+'  env='+d.env;
    document.getElementById('modeinfo').textContent = d.det?'[检测中 BPM]':'[无信号 BPM=0]';
    bpmHist.push(d.bpm>0?d.bpm:0); if(bpmHist.length>240) bpmHist.shift();
    drawWave(d.wave); drawBPM();
  }catch(e){}
}
setInterval(tick, 60);
</script></body></html>'''
print('HTML_PAGE =', len(HTML_PAGE), 'chars')

In [ ]:
\
import threading, json, webbrowser, socket
from collections import deque
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

# ---- 共享状态 ----
RING_MAX = 46300          # 环形保留 ~1s 采样
RING = deque(maxlen=RING_MAX)
LOCK = threading.Lock()
STATE = {'running': True, 'bpm': 0.0, 'swing': 0, 'env': 0, 'mode': MODE, 'det': False}

def _capture():
    """后台持续采 DMA -> 左声道 -> 环形缓冲, 同时刷新 BPM/swing/env"""
    ctr = 0
    while True:
        try:
            l = grab_block()
            swing = int(l.max() - l.min())
            if STATE['running']:
                with LOCK:
                    RING.extend(l.tolist())
                ctr += 1
                STATE['env'] = read_reg(0x60)
                _b = read_bpm()
                STATE['bpm'] = _b if (0 < _b < 260) else 0.0
                STATE['det'] = STATE['bpm'] > 0
            STATE['swing'] = swing   # 反映当前实时幅值
        except Exception:
            pass

threading.Thread(target=_capture, daemon=True).start()

# ---- HTTP 服务 ----
HOST = '0.0.0.0'; PORT = 8899
class H(BaseHTTPRequestHandler):
    def log_message(self, *a): pass
    def _json(self, obj):
        body = json.dumps(obj).encode(); 
        self.send_response(200); self.send_header('Content-Type','application/json')
        self.send_header('Content-Length', str(len(body))); self.end_headers(); self.wfile.write(body)
    def do_GET(self):
        if self.path.startswith('/api/state'):
            with LOCK: w = list(RING)
            st = min(len(w), 1600); step = max(1, len(w)//st) if w else 1
            wave = w[::step][-st:] if w else []
            self._json({'running': STATE['running'], 'bpm': STATE['bpm'],
                        'swing': STATE['swing'], 'env': STATE['env'],
                        'mode': STATE['mode'], 'det': STATE['det'], 'wave': wave})
            return
        if self.path in ('/', '/index.html'):
            body = HTML_PAGE.encode()
            self.send_response(200); self.send_header('Content-Type','text/html; charset=utf-8')
            self.send_header('Content-Length', str(len(body))); self.end_headers(); self.wfile.write(body)
            return
        self.send_response(404); self.end_headers()
    def do_POST(self):
        if self.path == '/api/start': STATE['running'] = True
        elif self.path == '/api/stop': STATE['running'] = False
        self._json({'ok': True, 'running': STATE['running']})

srv = ThreadingHTTPServer((HOST, PORT), H)
threading.Thread(target=srv.serve_forever, daemon=True).start()

try:
    ip = socket.gethostbyname(socket.gethostname())
    if ip.startswith('127.'):
        import subprocess
        ip = subprocess.check_output(['hostname','-I']).decode().split()[0]
except Exception:
    ip = '<board-ip>'
url = 'http://%s:%d/' % (ip, PORT)
print('实时网页: ' + url)
print('(在本页面浏览器里复制打开即可; 也可点下面的链接, 若弹出空白窗口属正常)')
try: webbrowser.open(url)
except Exception: pass


# 用法
- **当前(无线)**: 保持 `MODE='MIC'`，依次跑完代码 cell(1~4)，打开打印的 URL。
- **等电脑转接线到货后**: 把 Cell1 的 `MODE` 改成 `'LINE_IN'` 重跑即可，codec 自动用官方 0x07 线路输入配置，其余 cell 不变。
- 网页上 **Start Recording/Stop Recording** 控制录音；波形最近 ~1s 实时滚动，右上显示 BPM / Swing / 输入模式。
- 若打不开，换用打印的完整 URL(含 IP)。
- 每次 Restart Kernel 后要把这 4 个代码 cell(含 HTML 定义 cell)都重新跑一遍。